# Writing Feedback Tool — NLP/LLM Pipeline & Evaluation Harness
**CS254 Group 6 — AI-Powered Writing Feedback Tool**
**Role: Yaw Firempong, NLP/LLM Lead**

This notebook covers my part of the project:
1. Structured, rubric-based prompt design for the feedback-generation LLM call
2. The feedback-generation pipeline (essay in → structured feedback out)
3. The evaluation harness comparing model output to human-assigned scores (QWK, correlation)
4. A template for the human-evaluation rubric (clarity / actionability / correctness)

Dataset: [ASAP-AES](https://www.kaggle.com/c/asap-aes) — download `training_set_rel3.tsv` and point `DATA_PATH` at it below.


In [ ]:
# pip install anthropic pandas numpy scikit-learn scipy --break-system-packages
import os
import re
import json
import time
import pandas as pd
import numpy as np


## 1. Load the ASAP-AES dataset

ASAP-AES ships as a tab-separated file with (among others) these columns:
`essay_id, essay_set, essay, rater1_domain1, rater2_domain1, domain1_score`.

Score **ranges differ across the 8 essay sets** (e.g. set 1 is roughly 2–12, set 2 is 1–6, etc.),
so for this pass I'm working within a single `essay_set` at a time rather than mixing scales.


In [ ]:
DATA_PATH = "data/training_set_rel3.tsv"  # <- update to your local path

df = pd.read_csv(DATA_PATH, sep="\t", encoding="latin-1")
print(df.shape)
df[["essay_id", "essay_set", "essay", "domain1_score"]].head()


## 2. Rubric-based structured prompt

The rubric matches what's in the proposal: **organization, development, mechanics, voice**.
For each category the model returns strengths, weaknesses, and 2–3 actionable suggestions,
plus a single holistic `overall_score_1_to_6` so we have something numeric to compare against
the human `domain1_score`.


In [ ]:
RUBRIC = {
    "organization": "Is there a clear structure -- intro, coherent paragraphing, logical flow, and a conclusion?",
    "development": "Is the argument/content well developed with specific evidence, examples, or reasoning?",
    "mechanics": "Are grammar, spelling, punctuation, and sentence-level mechanics correct?",
    "voice": "Is there a clear, consistent authorial voice and register appropriate to the task?",
}

FEEDBACK_SCHEMA = """{
  "overall_score_1_to_6": <int>,
  "categories": {
    "organization": {"strengths": [...], "weaknesses": [...], "suggestions": [...]},
    "development": {"strengths": [...], "weaknesses": [...], "suggestions": [...]},
    "mechanics": {"strengths": [...], "weaknesses": [...], "suggestions": [...]},
    "voice": {"strengths": [...], "weaknesses": [...], "suggestions": [...]}
  }
}"""

def build_prompt(essay_text: str) -> str:
    rubric_lines = "\n".join(f"- {k.title()}: {v}" for k, v in RUBRIC.items())
    return f"""You are an experienced writing tutor giving structured, actionable feedback on a student essay draft.

Rubric categories:
{rubric_lines}

For EACH rubric category, give:
- 1-2 strengths
- 1-2 weaknesses
- 2-3 concrete, actionable revision suggestions

Also give an overall_score_1_to_6 estimating overall essay quality on a 1 (weak) to 6 (strong)
holistic scale.

Respond with ONLY valid JSON matching this schema, no extra commentary, no markdown fences:
{FEEDBACK_SCHEMA}

Essay:
\"\"\"
{essay_text}
\"\"\"
"""

print(build_prompt("Sample essay text goes here.")[:600])


## 3. Feedback-generation call

Wraps the Anthropic API call, strips stray markdown fences the model sometimes adds, parses
JSON, and retries with backoff on transient failures or malformed JSON.

Set your key first: `export ANTHROPIC_API_KEY=...` (don't hardcode it in the notebook).


In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

MODEL = "claude-haiku-4-5-20251001"  # cheap/fast tier -- plenty for rubric-based feedback

def get_feedback(essay_text: str, model: str = MODEL, max_retries: int = 3) -> dict:
    prompt = build_prompt(essay_text)
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=2000,  # extra headroom so the JSON never gets cut off mid-response
                messages=[{"role": "user", "content": prompt}],
            )
            # Some models return multiple content blocks (e.g. a "thinking" block before
            # the actual text answer), so we search for the text block instead of
            # assuming it's always content[0].
            text_blocks = [b.text for b in resp.content if b.type == "text"]
            if not text_blocks:
                raise ValueError(f"No text block in response (got block types: {[b.type for b in resp.content]})")
            raw = text_blocks[0].strip()
            raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
            return json.loads(raw)
        except (json.JSONDecodeError, Exception) as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {max_retries} attempts: {last_err}")


## 4. Batch run over a sample

Pull a sample from a single `essay_set` (keeping the score scale consistent), run the pipeline,
and collect both the human score and the LLM's `overall_score_1_to_6` for each essay.


In [ ]:
SAMPLE_N = 20
ESSAY_SET = 1  # pick one set so human_score is on one consistent scale

sample_df = df[df.essay_set == ESSAY_SET].sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

results = []
for i, row in sample_df.iterrows():
    try:
        fb = get_feedback(row["essay"])
        results.append({
            "essay_id": row["essay_id"],
            "human_score": row["domain1_score"],
            "llm_score": fb.get("overall_score_1_to_6"),
            "feedback": fb,
        })
        print(f"[{i+1}/{SAMPLE_N}] essay {row['essay_id']}: human={row['domain1_score']} llm={fb.get('overall_score_1_to_6')}")
    except Exception as e:
        print(f"Failed on essay {row['essay_id']}: {e}")
    time.sleep(1)  # be polite to the API / stay under rate limits

results_df = pd.DataFrame(results)
results_df.head()


## 5. Evaluation harness — QWK & correlation

The human `domain1_score` for a given essay set and the model's `overall_score_1_to_6` live on
different scales, so I rescale the LLM score onto the human score's range before computing
Quadratic Weighted Kappa (QWK expects two sets of scores on the same discrete scale). Pearson
and Spearman correlation don't need rescaling.


In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import pearsonr, spearmanr

def rescale(series, target_min, target_max):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return series * 0 + target_min
    return (series - s_min) / (s_max - s_min) * (target_max - target_min) + target_min

human = results_df["human_score"].astype(float)
llm = results_df["llm_score"].astype(float)

llm_rescaled = rescale(llm, human.min(), human.max()).round().astype(int)
human_int = human.round().astype(int)

qwk = cohen_kappa_score(human_int, llm_rescaled, weights="quadratic")
pearson_r, pearson_p = pearsonr(human, llm)
spearman_r, spearman_p = spearmanr(human, llm)

print(f"Quadratic Weighted Kappa: {qwk:.3f}")
print(f"Pearson r:  {pearson_r:.3f}  (p={pearson_p:.3g})")
print(f"Spearman r: {spearman_r:.3f}  (p={spearman_p:.3g})")


## 6. Human-evaluation rubric template

For the small human-eval pass (team + 2–3 outside volunteers rating clarity, actionability,
correctness of the *feedback itself*, not the essay), this generates a CSV the raters can fill in.


In [ ]:
human_eval_rows = []
for _, r in results_df.iterrows():
    human_eval_rows.append({
        "essay_id": r["essay_id"],
        "clarity_1_5": "",
        "actionability_1_5": "",
        "correctness_1_5": "",
        "rater_name": "",
        "notes": "",
    })

human_eval_df = pd.DataFrame(human_eval_rows)
human_eval_df.to_csv("human_eval_template.csv", index=False)
human_eval_df.head()


## 7. Inspect one example end-to-end


In [ ]:
example = results_df.iloc[0]
print(f"Essay {example['essay_id']}  |  human_score={example['human_score']}  llm_score={example['llm_score']}\n")
print(json.dumps(example["feedback"], indent=2))


## 8. Save outputs

Keep the raw results (including full structured feedback) so Darrell can pull the quantitative
metrics into the shared evaluation report, and so the ethics audit has real feedback text to
review for bias.


In [ ]:
results_df.to_json("llm_feedback_results.json", orient="records", indent=2)
results_df[["essay_id", "human_score", "llm_score"]].to_csv("llm_vs_human_scores.csv", index=False)
print("Saved: llm_feedback_results.json, llm_vs_human_scores.csv, human_eval_template.csv")
